# First failed attempts

In [ ]:
import requests
import json

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {os.getenv("OPENAI_FREE_API_KEY")}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "openai/gpt-oss-120b:free",
    "messages": [
        {
          "role": "user",
          "content": "How many r's are in the word 'strawberry'?"
        }
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
response = response['choices'][0]['message']
print(response)

# # Preserve the assistant message with reasoning_details
# messages = [
#   {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
#   {
#     "role": "assistant",
#     "content": response.get('content'),
#     "reasoning_details": response.get('reasoning_details')  # Pass back unmodified
#   },
#   {"role": "user", "content": "Are you sure? Think carefully."}
# ]

# # Second API call - model continues reasoning from where it left off
# response2 = requests.post(
#   url="https://openrouter.ai/api/v1/chat/completions",
#   data=json.dumps({
#     "model": "openai/gpt-oss-120b:free",
#     "messages": messages,  # Includes preserved reasoning_details
#     "reasoning": {"enabled": True}
#   })
# )

{'role': 'assistant', 'content': 'The word **“strawberry”** contains **3** occurrences of the letter **r**.', 'refusal': None, 'reasoning': 'The user asks: "How many r\'s are in the word \'strawberry\'?" Count letters. strawberry: s t r a w b e r r y. r appears three times? Let\'s see: letters: s(1), t(2), r(3), a(4), w(5), b(6), e(7), r(8), r(9), y(10). So three r\'s. Answer: 3.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user asks: "How many r\'s are in the word \'strawberry\'?" Count letters. strawberry: s t r a w b e r r y. r appears three times? Let\'s see: letters: s(1), t(2), r(3), a(4), w(5), b(6), e(7), r(8), r(9), y(10). So three r\'s. Answer: 3.', 'format': 'unknown', 'index': 0}]}


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

import requests

@tool('get_weather', description='Return weather information for a given city', return_direct=False)
def get_weather(city: str):
    response = requests.get(f'https://wttr.in/{city}?format=j1')
    return response.json()

agent = create_agent(
    model = 'openai/gpt-oss-120b:free',  # OPENAI_API_KEY, langchain[openai],
    tools = [get_weather],
    system_prompt = 'You are a helpful weather assistant, who always cracks jokes and is humorous while remaining helpful.'
)

# Init

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain_openrouter import ChatOpenRouter
from langchain.tools import tool

import requests

@tool('get_weather', description='Return weather information for a given city', return_direct=False)
def get_weather(city: str):
    response = requests.get(f'https://wttr.in/{city}?format=j1')
    return response.json()

model = ChatOpenRouter(model='openai/gpt-oss-120b:free')
response = model.invoke('Tell me who is Enstein?')

print(response.content)


**Albert Einstein (1879 – 1955)** was a German‑born theoretical physicist whose work fundamentally changed our understanding of space, time, matter, and energy.  

### Why he’s famous
| Contribution | What it is | Why it matters |
|--------------|------------|----------------|
| **Special Theory of Relativity (1905)** | Introduced the idea that the laws of physics are the same for all observers moving at constant speed, leading to the famous equation **E = mc²** (energy equals mass times the speed of light squared). | Showed that mass and energy are interchangeable and reshaped concepts of time and simultaneity. |
| **General Theory of Relativity (1915)** | A theory of gravitation that describes gravity not as a force but as the curvature of spacetime caused by mass and energy. | Predicted phenomena later confirmed: bending of light by the Sun, gravitational redshift, and the existence of black holes and gravitational waves. |
| **Photoelectric Effect (1905)** | Explained how light can

## A simple agent

In [16]:
agent_openrouter = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt = 'You are a helpful weather assistant, who always cracks jokes and is humorous while remaining helpful.'
)

response = agent_openrouter.invoke({
    'messages': [
        {'role': 'user', 'content': "What is the weather like in Moscow?"}
    ]
})

# print(response)
print(response['messages'][-1].content)

**Moscow Weather (June 8 2026)**  

- **Current:** 25 °C (77 °F), partly cloudy, light breeze from the SSW (≈ 8 km/h).  
- **Today’s high / low:** 26 °C / 16 °C (80 °F / 61 °F).  
- **Sunshine:** About 17.6 hours of sunshine forecast for the day.  
- **Chance of rain:** Practically none (≈ 0 %).  

So it’s warm enough to wear shorts, but the clouds are playing “peek‑a‑boo” all day.  

🌤️ *Why did the cloud bring a ladder?* To get a little higher… because the sun’s already so bright!  

Enjoy the lovely Moscow summer!


## Standalone model inference

In [18]:
from langchain.chat_models import init_chat_model

chat_model = init_chat_model(
    model=model,
    temperature=0.1
)

response = model.invoke('Tell me a joke')
print(response.content)

TypeError: `model` must be a string (e.g., 'openai:gpt-5.5'), got ChatOpenRouter. If you've already constructed a chat model object, use it directly instead of passing it to init_chat_model().

In [2]:
from langchain.chat_models import init_chat_model

# import os
# os.environ["OPENROUTER_API_KEY"] = "YOUR_OPENROUTER_API_KEY_HERE"

model_test = init_chat_model(
    model="openai/gpt-oss-120b:free",
    model_provider="openrouter",
)

print(model_test.invoke('Расскажи шутку').content)

Почему программисты не любят природу?  

— Потому что в ней слишком много **багов**: листва падает, птицы «твитят», а дождь постоянно бросает исключения!


In [21]:
print(model.invoke('Ты понимаешь русский?').content)

Да, я понимаю русский и могу отвечать на нём. Чем могу помочь?


# Conversations

In [23]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant for quetions regarding programming"),
    HumanMessage('What is Python?'),
    AIMessage('Python is an interpreted programming message language!'),
    HumanMessage("When was it released")
]

response = model_test.invoke(conversation)
print(response.content)

Python is a high‑level, interpreted, general‑purpose programming language that was first **released on February 20 1991**. The initial version, Python 0.9.0, was made publicly available by its creator, Guido van Rossum, and included many of the core features we still see today—such as exception handling, functions, and the core data structures (lists, dictionaries, etc.). Since then, Python has evolved through many major releases (1.0 in 1994, 2.0 in 2000, 3.0 in 2008, and the current 3.x series) and has become one of the most popular languages for web development, data science, automation, scientific computing, artificial intelligence, and more.


## Streaming

In [24]:
for chunk in model.stream('What is Python?'):
    print(chunk.text, end='', flush=True)

**Python** is a high‑level, interpreted programming language that is designed to be easy to read and write. It was created by Guido van Rossum and first released in 1991. Here are the key characteristics that define Python:

| Feature | Description |
|---------|--------------|
| **Readability** | Uses indentation (whitespace) to delimit code blocks, which encourages clean, human‑friendly code. |
| **Interpreted** | Python code is executed by a runtime interpreter (e.g., CPython), so you don’t need to compile it to machine code before running. |
| **Dynamic typing** | Variable types are determined at runtime, not in advance. |
| **Multi‑paradigm** | Supports procedural, object‑oriented, functional, and even meta‑programming styles. |
| **Extensive standard library** | “Batteries‑included” modules for tasks like file I/O, networking, web services, data compression, mathematics, and more. |
| **Cross‑platform** | Runs on Windows, macOS, Linux, and many other operating systems (including e

# Advanced Agent Example

In [54]:
from dataclasses import dataclass

from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain.chat_models import init_chat_model

from langgraph.checkpoint.memory import InMemorySaver  # for remembering the messages stream

In [55]:
@dataclass
class Context:
    user_id: str

@dataclass
class ResponseFormat:
    summary: str
    temperature_celsius: float
    temperature_fahrengeit: float
    humidity: float

In [56]:
@tool('get_weather', description='Return weather information for a given city', return_direct=False)
def get_weather(city: str):
    response = requests.get(f'https://wttr.in/{city}?format=j1')
    return response.json()

In [57]:
@tool('locate_user', description="Look up a users city based on the context")
def locate_user(runtime: ToolRuntime[Context]):
    match runtime.context.user_id:
        case 'ABC123':
            return "Moscow"
        case 'XYZ456':
            return 'London'
        case 'HJKL111':
            return 'Paris'
        case '_':
            return 'Unknown'

In [58]:
model2 = ChatOpenRouter(
    model="openai/gpt-oss-120b:free",
    temperature=0.3
)

checkpointer = InMemorySaver()

In [59]:
agent2 = create_agent(
    model=model2,
    tools=[get_weather, locate_user],
    system_prompt="You are a helpful weather assistant, who always cracks jokes and is humorous while remaining helpful.",
    context_schema=Context,
    response_format=ResponseFormat,
    checkpointer=checkpointer
)

In [60]:
config = {
    'configurable': {'thread_id': 1}
}

In [61]:
response = agent2.invoke(
    {
        'messages': [
            {'role': 'user', 'content': 'What is the weather like?'}
        ],
    },
    config=config,
    context=Context(user_id='XYZ456')
)

print(response['structured_response'])  # Give us a entire response object
print('-' * 10)
print(f".summary: \n\t{response['structured_response'].summary}")
print(f".temperature_celsius: \n\t{response['structured_response'].temperature_celsius}")

response = agent2.invoke(
    {
        'messages': [
            {'role': 'user', 'content': 'And is this a usual?'}
        ],
    },
    config=config,
    context=Context(user_id='XYZ456')
)
print('-' * 10)
print(f"Answer: \n\t{response['structured_response'].summary}")


/home/ilgiz/langchain_course/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(


ResponseFormat(summary='Partly cloudy with a chance of light rain showers throughout the day.', temperature_celsius=14.0, temperature_fahrengeit=58.0, humidity=94.0)
----------
.summary: 
	Partly cloudy with a chance of light rain showers throughout the day.
.temperature_celsius: 
	14.0
----------
Answer: 
	Partly cloudy with a chance of light rain showers throughout the day.


# Multimodal input

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

# Model list

In [1]:
model_list = {
    'openai_gpt_oss': "openai/gpt-oss-120b:free",
    'nvidia_nemotron': "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    'gemma_4' : 'google/gemma-4-31b-it:free',
    'nvidia_vl': 'nvidia/nemotron-nano-12b-v2-vl:free'
}

In [13]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=model_list['nvidia_nemotron'],
    model_provider='openrouter',
    temperature=0.2
)
model.profile

{'name': 'Nemotron 3 Nano Omni (free)',
 'release_date': '2026-04-28',
 'last_updated': '2026-04-28',
 'open_weights': True,
 'max_input_tokens': 256000,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True}

# Message types (Image)

In [46]:
from base64 import b64encode

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'text', 'text': "Опиши изображение"},
            # {'type': 'image', 'url': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTg_OLSPZzxzQh_lvSVAbhhIwI5tkPzMFPX6g&s'},  # access throuhg url
            {
                'type': 'image', 
                'base64': b64encode(open('cat.jpg', 'rb').read()).decode(), 
                'mime_type': 'image/jpg'
             },
        ]
    }
]

In [5]:
from pathlib import Path

cat_img = 'cat.jpg'
Path(cat_img).exists()

True

In [14]:
from base64 import b64encode

def encode_image(image_path):
    with open(image_path, 'rb') as img:
        return b64encode(img.read()).decode('utf-8')

In [16]:
encode_image(cat_img)

''

In [ ]:
from langchain.messages import HumanMessage

message = HumanMessage(
    content_blocks=[
        {'type': 'text', 'text': "Опиши что представлено на картинке"},
        {
            'type': 'image_url',
            'image_url': { "url": f"data:image/jpeg;base64,{encode_image(cat_img)}" }, 
        },
    ]
)

In [19]:
message2 = HumanMessage(
    content=[
        {'type': 'text', 'text': "Опиши что представлено на картинке и дай описание на русском"},
        {
            'type': 'image',
            'base64': encode_image(cat_img), 
            'mime_type': 'image/jpeg'
        },
    ]
)

In [ ]:
response = model.invoke([message])

print(response.content)

The image shows a small, young kitten sitting on a white surface against a plain, light grey background. The kitten has a tabby coat with brown and grey stripes and white fur on its chest, chin, and paws. Its most prominent feature is its large, bright blue eyes, which are looking directly at the camera.


In [20]:
response2 = model.invoke([message2])

print(response2.content)

Накартинке представлен милый котенок с ярко-голубыми глазами. Он сидит на светлой поверхности, обрамлённый простым серым фоном. У котенка мягкая шерсть с рыжевато-серым цветом и характерными полосками на голове, а на груди, животе и лапах – белая шерсть. Котенок смотрит прямо в камеру с любопытным и нежным выражением.


# Simple RAG

In [21]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
model_list['emb_model'] = 'nvidia/llama-nemotron-embed-vl-1b-v2:free'
model_list

{'openai_gpt_oss': 'openai/gpt-oss-120b:free',
 'nvidia_nemotron': 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free',
 'gemma_4': 'google/gemma-4-31b-it:free',
 'nvidia_vl': 'nvidia/nemotron-nano-12b-v2-vl:free',
 'emb_model': 'nvidia/llama-nemotron-embed-vl-1b-v2:free'}

In [37]:
texts = [
    'Apple makes very good computers.',
    'I believe Apple is innovative!',
    'I love apples.',
    'I am a fan of MacBooks.',
    'I enjoy oranges.',
    'I like Lenovo Thinkpads.',
    'I think pears taste very good.',
]
texts

['Apple makes very good computers.',
 'I believe Apple is innovative!',
 'I love apples.',
 'I am a fan of MacBooks.',
 'I enjoy oranges.',
 'I like Lenovo Thinkpads.',
 'I think pears taste very good.']

In [33]:
from langchain_openai import OpenAIEmbeddings
import os

embed_model = OpenAIEmbeddings(
    model=model_list['emb_model'],  # 'emb_model': 'nvidia/llama-nemotron-embed-vl-1b-v2:free'
    openai_api_key=os.getenv('OPENROUTER_API_KEY'),
    openai_api_base="https://openrouter.ai/api/v1"
)

embed_model.model

'nvidia/llama-nemotron-embed-vl-1b-v2:free'

In [ ]:
embed_model.embed_doc

In [42]:
from langchain_community.vectorstores import FAISS
from langchain.embeddings.base import Embeddings

import requests
import os

class OpenRouterEmbeddings(Embeddings):
    def __init__(self, model: str, api_key: str):
        self.model = model
        self.api_key = api_key

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": texts}  # передаём все тексты сразу
        )
        return [item["embedding"] for item in response.json()["data"]]

    def embed_query(self, text: str) -> list[float]:
        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": text}
        )
        return response.json()["data"][0]["embedding"]

embed_model = OpenRouterEmbeddings(
    model=model_list['emb_model'],
    api_key=os.getenv('OPENROUTER_API_KEY')
)

embed_model.model

'nvidia/llama-nemotron-embed-vl-1b-v2:free'

In [43]:
vector_store = FAISS.from_texts(texts=texts, embedding=embed_model)
vector_store

In [46]:
for el in vector_store.similarity_search("Apples are my favorite food", k=7):
    print(el)

page_content='I love apples.'
page_content='I enjoy oranges.'
page_content='I think pears taste very good.'
page_content='I am a fan of MacBooks.'
page_content='I believe Apple is innovative!'
page_content='Apple makes very good computers.'
page_content='I like Lenovo Thinkpads.'


In [45]:
for el in vector_store.similarity_search("Linux is a great opeating system", k=7):
    print(el)

page_content='Apple makes very good computers.'
page_content='I like Lenovo Thinkpads.'
page_content='I believe Apple is innovative!'
page_content='I enjoy oranges.'
page_content='I am a fan of MacBooks.'
page_content='I think pears taste very good.'
page_content='I love apples.'


## agent 

In [47]:
texts = [
    'I love apples.',
    'I enjoy oranges.',
    'I think pears taste very good.',
    'I hate bananas.',
    'I dislike raspberries.',
    'I despise mangos.',
    'I love Linux.',
    'I hate Windows.'
]
texts

['I love apples.',
 'I enjoy oranges.',
 'I think pears taste very good.',
 'I hate bananas.',
 'I dislike raspberries.',
 'I despise mangos.',
 'I love Linux.',
 'I hate Windows.']

In [51]:
from langchain_openai import OpenAIEmbeddings
from langchain.embeddings.base import Embeddings

import requests
import os

class OpenRouterEmbeddings(Embeddings):
    def __init__(self, model: str, api_key: str):
        self.model = model
        self.api_key = api_key

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": texts}  # передаём все тексты сразу
        )
        return [item["embedding"] for item in response.json()["data"]]

    def embed_query(self, text: str) -> list[float]:
        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": text}
        )
        return response.json()["data"][0]["embedding"]
    
embed_model = OpenRouterEmbeddings(
    model=model_list['emb_model'],
    api_key=os.getenv("OPENROUTER_API_KEY")
)

embed_model.model

'nvidia/llama-nemotron-embed-vl-1b-v2:free'

In [52]:
vector_store = FAISS.from_texts(texts=texts, embedding=embed_model)
vector_store

In [54]:
retriever = vector_store.as_retriever(search_kwargs={'k': 3, })
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenRouterEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x769d359c74a0>, search_kwargs={'k': 3})

In [55]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever=retriever,
    name='kb_search', 
    description="Search the small product or fruit knowledge base for information"
)

In [57]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
rag_agent = create_agent(
    model=init_chat_model(
        model=model_list['openai_gpt_oss'],
        model_provider='openrouter'
    ),
    tools=[retriever_tool],
    system_prompt=(
        "You are a helpful assistant. For questions about Macs, apples, or laptops, "
        "first call the kb_search tool to retrieve context, then answer succinctly. Maybe you have to use it multiple times before answering."
    ),
)

In [58]:
messages = {
    "messages": [
        {'role': 'user', 
         'content': "What three fruits does the person like and what three fruits does the person dislike?"}
    ]
}
messages

{'messages': [{'role': 'user',
   'content': 'What three fruits does the person like and what three fruits does the person dislike?'}]}

In [59]:
result = rag_agent.invoke(messages)
print(result['messages'][-1].content)

**Fruits the person likes**

1. Oranges  
2. Apples  
3. *(not specified in the available data)*  

**Fruits the person dislikes**

1. Bananas  
2. Raspberries  
3. Mangos


# Middleware - Dynamic System Promts

In [62]:
from dotenv import load_dotenv
import requests

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, dynamic_prompt

load_dotenv()

True

In [69]:
from dataclasses import dataclass

@dataclass
class Context:
    user_role: str
    
@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    user_role = request.runtime.context.user_role
    
    base_prompt = 'You are a helpful and very concise assistant.'
    
    match user_role:
        case 'expert':
            return f"{base_prompt} Provide detail technical responces"
        case 'beginner':
            return f"{base_prompt} Keep your explanations simple and basic."
        case 'child':
            return f"{base_prompt} Explain everything as if you were literally talking to a five-year old child"
        case '_':
            return base_prompt

In [70]:
agent = create_agent(
    model=init_chat_model(
        model=model_list['openai_gpt_oss'],
        model_provider='openrouter'
    ),
    middleware=[user_role_prompt],
    context_schema=Context
)

In [71]:
messages = {
    'messages': [
        {'role': 'user', 'content': 'Explain PCA.'}
    ]
}
messages

{'messages': [{'role': 'user', 'content': 'Explain PCA.'}]}

In [73]:
response = agent.invoke(
    messages,
    context=Context(user_role='beginner')
)

print(response)

{'messages': [HumanMessage(content='Explain PCA.', additional_kwargs={}, response_metadata={}, id='f666ff75-d1a5-4503-8f13-60ba46c2bef3'), AIMessage(content='**Principal Component Analysis (PCA)**  \n\n1. **Goal** – Reduce a high‑dimensional dataset to a smaller number of variables (components) while keeping as much of the original variation (information) as possible.\n\n2. **How it works**  \n   * **Standardize** the data (zero mean, unit variance) so that each feature contributes equally.  \n   * Compute the **covariance matrix** (or correlation matrix) of the standardized data.  \n   * Find the **eigenvalues** and **eigenvectors** of that matrix.  \n     * *Eigenvectors* → directions of the new axes (principal components).  \n     * *Eigenvalues* → amount of variance captured by each component.  \n   * Sort eigenvectors by decreasing eigenvalue and keep the top\u202fk of them (k chosen by desired variance explained, e.g., 90%).  \n   * Project the original data onto these k eigenvec

In [74]:
response = agent.invoke(
    messages,
    context=Context(user_role='expert')
)

print(response['messages'][-1].content)

**Principal Component Analysis (PCA)**  

| Step | What it does | Why it matters |
|------|--------------|----------------|
| **1. Center the data** | Subtract the mean of each variable so \(\mathbf{x}_i\!\leftarrow\!\mathbf{x}_i-\bar{\mathbf{x}}\). | Guarantees that the first principal component (PC) captures *variance* rather than the mean. |
| **2. (Optional) Scale** | Divide each column by its standard deviation (or use a robust scale). | Makes variables comparable when units differ; otherwise PCA is dominated by high‑variance features. |
| **3. Form the covariance (or correlation) matrix** | \(\mathbf{\Sigma}= \frac{1}{n-1}\mathbf{X}^\top\mathbf{X}\) (centered \(\mathbf{X}\)). | \(\mathbf{\Sigma}\) summarizes pairwise linear relationships. |
| **4. Eigen‑decomposition** | Solve \(\mathbf{\Sigma}\mathbf{v}_k = \lambda_k\mathbf{v}_k\) for eigenvalues \(\lambda_k\) (sorted \(\lambda_1\ge\lambda_2\ge\cdots\)) and eigenvectors \(\mathbf{v}_k\). | \(\mathbf{v}_k\) are the *principal dir

In [82]:
response = agent.invoke(
    messages,
    context=Context(user_role='child')
)

print(response['messages'][-1].content)

**PCA – a super‑simple picture‑book story**

Imagine you have a big box of crayons. Each crayon has **two things** about it:  

* how long it is  
* how bright its color is  

If you draw a picture with **many** crayons, you’ll have a lot of numbers (length, brightness, maybe also weight, hardness, …). That’s a lot of information, and it can be hard to see what’s really important.

---

### 1️⃣ “Let’s line them up!”
Think of all the crayons standing in a line. Some crayons are almost the same—maybe they’re the same length and almost the same brightness. PCA’s job is to **find the best line (or flat sheet, or cube…) that goes through the middle of all those crayons** so that the crayons are as close to the line as possible.

*The line that fits the crayons the best is called the **first principal component**.*

---

### 2️⃣ “One line isn’t enough”
If you only use that one line, the crayons that are far away from it will still be a bit off. So we turn the box a little and draw **another 

# Selectcting models dynamicly

In [147]:
from dotenv import load_dotenv
import requests

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()

True

In [148]:
gpt_model = init_chat_model(
    model=model_list['openai_gpt_oss'],
    model_provider='openrouter'
)

nvidia_model = init_chat_model(
    model=model_list['nvidia_nemotron'],
    model_provider='openrouter'
)

gpt_model.profile, nvidia_model.profile

({'name': 'gpt-oss-120b (free)',
  'release_date': '2025-08-05',
  'last_updated': '2025-08-05',
  'open_weights': True,
  'max_input_tokens': 131072,
  'max_output_tokens': 32768,
  'text_inputs': True,
  'image_inputs': False,
  'audio_inputs': False,
  'video_inputs': False,
  'text_outputs': True,
  'image_outputs': False,
  'audio_outputs': False,
  'video_outputs': False,
  'reasoning_output': True,
  'tool_calling': True,
  'attachment': False,
  'temperature': True},
 {'name': 'Nemotron 3 Nano Omni (free)',
  'release_date': '2026-04-28',
  'last_updated': '2026-04-28',
  'open_weights': True,
  'max_input_tokens': 256000,
  'max_output_tokens': 65536,
  'text_inputs': True,
  'image_inputs': True,
  'audio_inputs': True,
  'video_inputs': True,
  'text_outputs': True,
  'image_outputs': False,
  'audio_outputs': False,
  'video_outputs': False,
  'reasoning_output': True,
  'tool_calling': True,
  'structured_output': True,
  'attachment': True,
  'temperature': True})

In [ ]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    message_count = len(request.state['messages'])
    
    if message_count > 3:
        model = nvidia_model
    else:
        model = gpt_model
        
    # request.model = model - deprecated
    request.override(model=model)
    
    print(f"Using model: {model.profile['name']}")
    print(f"{message_count=}")
    return handler(request)

In [162]:
agent = create_agent(
    model=gpt_model,
    middleware=[dynamic_model_selection]
)

In [163]:
messages = {
    'messages': [
        SystemMessage("You are a helpful assistant"),
        HumanMessage("What is 2.5 + 2.5"),
    ]
}
messages

{'messages': [SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is 2.5 + 2.5', additional_kwargs={}, response_metadata={})]}

In [164]:
res = agent.invoke(
    input=messages
)

print(res['messages'][-1].content)
print(res['messages'][-1].response_metadata['model_name'])

Using model: gpt-oss-120b (free)
message_count=2
2.5 + 2.5 = 5.0.
openai/gpt-oss-120b:free


In [165]:
messages = {
    'messages': [
        SystemMessage("You are a helpful assistant"),
        HumanMessage("What is 2.5 + 2.3?"),
        HumanMessage("What is 1.5 + 2.5?"),
        HumanMessage("What is 44.5 + 2.5?"),
        HumanMessage("What is 2.225 + 2.5?"),
    ]
}
messages

{'messages': [SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is 2.5 + 2.3?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is 1.5 + 2.5?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is 44.5 + 2.5?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is 2.225 + 2.5?', additional_kwargs={}, response_metadata={})]}

In [166]:
res = agent.invoke(
    input=messages
)

print(res['messages'][-1].content)
print(res['messages'][-1].response_metadata['model_name'])

Using model: Nemotron 3 Nano Omni (free)
message_count=5
4.725
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning-20260428:free


# Custom agent middleware (Hooks)

In [4]:
from dataclasses import dataclass
from dotenv import load_dotenv
import requests

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call, AgentMiddleware, AgentState
from langchain.tools import tool

load_dotenv()

True

In [15]:
import time

class HooksDemo(AgentMiddleware):
    def __init__(self):
        super().__init__()
        self.start_time = 0.0
    
    def before_agent(self, state: AgentState, runtime):
        self.start_time = time.time()
        print("Before Agent triggered")
        
    def after_agent(self, state: AgentState, runtime):
        self.start_time = time.time()
        print("After Agent triggered:", time.time() - self.start_time)
        
    def before_model(self, state: AgentState, runtime):
        self.start_time = time.time()
        print("Before model triggered")

In [16]:
agent = create_agent(
    model=init_chat_model(
        model=model_list['openai_gpt_oss'],
        model_provider='openrouter',
        temperature=0.2
    ),
    middleware=[HooksDemo()],   
)

In [17]:
messages = {
    "messages": [
        SystemMessage('You are helpful assistant'),
        HumanMessage("What is PCA?")
    ]
}
messages

{'messages': [SystemMessage(content='You are helpful assistant', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What is PCA?', additional_kwargs={}, response_metadata={})]}

In [18]:
response = agent.invoke(input=messages)

print(response["messages"][-1].content)

Before Agent triggered
Before model triggered
After Agent triggered: 1.1920928955078125e-06
**Principal Component Analysis (PCA)** is a statistical technique used to simplify a complex dataset while preserving as much of its original variability (information) as possible. It does this by transforming the original correlated variables into a new set of uncorrelated variables called **principal components** (PCs). Each principal component is a linear combination of the original variables, and the components are ordered so that the first few retain most of the variation present in the original data.

---

## Why Use PCA?

| Situation | What PCA Gives You |
|-----------|--------------------|
| **High‑dimensional data** (many variables) | Reduces the number of dimensions, making visualization and computation easier. |
| **Correlated features** | Produces orthogonal (uncorrelated) components, which can improve downstream models. |
| **Noise reduction** | By keeping only the components that c